In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install streamlit pyngrok plotly pandas numpy matplotlib seaborn wordcloud streamlit-option-menu

In [ ]:
%%writefile dashboard.py

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from streamlit_option_menu import option_menu
from PIL import Image
import os
st.set_page_config(
    page_title="Healthcare Literature Dashboard",
    page_icon="📚",
    layout="wide"
)

# =========================
# CUSTOM CSS
# =========================
st.markdown(
    """
    <style>
    .main {
        background-color: #0E1117;
    }

    .stMetric {
        background-color: #1E1E1E;
        padding: 15px;
        border-radius: 15px;
        border: 1px solid #333;
    }

    /* Metric Label */
    .stMetric label {
        color: #BBBBBB !important;
    }

    /* Metric Number */
    [data-testid="stMetricValue"] {
        color: #00FFAA !important;   /* Change this color */
        font-size: 40px;
        font-weight: bold;
    }

    h1, h2, h3 {
        color: white;
    }
    </style>
    """,
    unsafe_allow_html=True
)


DATA_PATH = "/content/drive/MyDrive/dashboard_files"

# =========================
# LOAD DATA
# =========================

url = "/content/drive/MyDrive/final_dataset.csv"
df = pd.read_csv(url)
papers_df = pd.read_csv(f"{DATA_PATH}/healthcare_papers_enriched.csv")
top_keywords = pd.read_csv(f"{DATA_PATH}/top_keywords_tfidf.csv")
lda_topics = pd.read_csv(f"{DATA_PATH}/lda_topic_words.csv")

# =========================
# SIDEBAR MENU
# =========================

with st.sidebar:
    selected = option_menu(
        menu_title="Research Dashboard",
        options=[
            "Overview",
            "Research Trends",
            "Topic Modeling",
            "Keyword Analysis",
            "Author Analysis",
            "Research Gaps",
            "Emerging Domain Detection",
            "Paper Explorer",
            "Interactive Visuals"
        ],
        icons=[
            "house",
            "graph-up",
            "diagram-3",
            "search",
            "people",
            "exclamation-triangle",
            "rocket",
            "book",
            "bar-chart"
        ],
        default_index=0
    )
# OVERVIEW PAGE
# =========================

if selected == "Overview":

    st.title("📚 Healthcare Research Literature Dashboard")

    total_papers = len(papers_df)
    total_topics = papers_df['dominant_topic_lda'].nunique()
    avg_authors = round(papers_df['n_authors'].mean(), 2)
    total_categories = papers_df['categories'].nunique()

    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Total Papers", total_papers)
    col2.metric("Total Topics", total_topics)
    col3.metric("Avg Authors", avg_authors)
    col4.metric("Research Categories", total_categories)

    st.markdown("---")

    st.subheader("Dataset Preview")
    st.dataframe(papers_df.head(20), use_container_width=True)

    st.subheader("Research Distribution by Year")

    yearly = papers_df.groupby('year').size().reset_index(name='count')

    fig = px.line(
        yearly,
        x='year',
        y='count',
        markers=True,
        title='Research Publications Over Time'
    )
    st.plotly_chart(fig, use_container_width=True)
    st.markdown("---")

    st.subheader("ArXiv Category Distribution — Healthcare Papers")

    html_path = os.path.join(DATA_PATH, "category_distribution.html")

    if os.path.exists(html_path):
        with open(html_path, "r", encoding="utf-8") as f:
            html_data = f.read()

        st.components.v1.html(html_data, height=600, scrolling=True)

    st.markdown("---")

    st.subheader("Top ArXiv Categories — Publication Trends")

    html_path2 = os.path.join(DATA_PATH, "category_trends.html")

    if os.path.exists(html_path2):
        with open(html_path2, "r", encoding="utf-8") as f:
            html_data2 = f.read()

        st.components.v1.html(html_data2, height=600, scrolling=True)


# =========================
# =========================
# RESEARCH TRENDS
# =========================

elif selected == "Research Trends":

    st.title("📈 Research Trends Analysis")

    yearly = papers_df.groupby('year').size().reset_index(name='papers')

    fig = px.area(
        yearly,
        x='year',
        y='papers',
        title='Publication Growth Over Time'
    )

    st.plotly_chart(fig, use_container_width=True)

    st.subheader("Topic Distribution")

    topic_counts = papers_df['dominant_topic_lda'].value_counts().reset_index()
    topic_counts.columns = ['Topic', 'Count']

    fig2 = px.bar(
        topic_counts,
        x='Topic',
        y='Count',
        color='Count',
        title='Dominant Topic Distribution'
    )
    st.plotly_chart(fig2, use_container_width=True)
    st.markdown("---")

    st.subheader("Topic Trends Over Time")

    trend_html = os.path.join(DATA_PATH, "topic_trends_over_time.html")

    if os.path.exists(trend_html):
        with open(trend_html, "r", encoding="utf-8") as f:
            html_data = f.read()

        st.components.v1.html(html_data, height=700, scrolling=True)



# =========================
# TOPIC MODELING
# =========================

elif selected == "Topic Modeling":

    st.title("🧠 Topic Modeling Insights")

    topic_labels = lda_topics.groupby('topic')['word'].apply(lambda x: ', '.join(x.head(10)))

    for topic, words in topic_labels.items():
        st.markdown(f"### {topic}")
        st.info(words)

    st.markdown("---")

    st.subheader("Topic Confidence Distribution")

    fig = px.histogram(
        papers_df,
        x='topic_confidence',
        nbins=30,
        title='Topic Confidence Scores'
    )
    st.plotly_chart(fig, use_container_width=True)
    st.markdown("---")

    st.subheader("BERTopic Visualization")

    bertopic_html = os.path.join(DATA_PATH, "bertopic_barchart.html")

    if os.path.exists(bertopic_html):
        with open(bertopic_html, "r", encoding="utf-8") as f:
            bertopic_data = f.read()

        st.components.v1.html(bertopic_data, height=700, scrolling=True)


# =========================
# KEYWORD ANALYSIS
# =========================

elif selected == "Keyword Analysis":

    st.title("🔍 Keyword Analysis")

    st.subheader("Top TF-IDF Keywords")

    top_20 = top_keywords.head(20)

    fig = px.bar(
        top_20,
        x='tfidf_score',
        y='keyword',
        orientation='h',
        title='Top Keywords'
    )

    st.plotly_chart(fig, use_container_width=True)

    image_path = f"{DATA_PATH}/wordcloud_healthcare.png"

    if os.path.exists(image_path):
        st.subheader("Healthcare WordCloud")
        image = Image.open(image_path)
        st.image(image, use_container_width=True)
    comparison_path = os.path.join(DATA_PATH, "keywords_comparison.png")

    if os.path.exists(comparison_path):
        st.markdown("---")
        st.subheader("Keyword Comparison Chart")

        comparison_img = Image.open(comparison_path)

        st.image(comparison_img, use_container_width=True)

# =========================
# AUTHOR ANALYSIS
# =========================

elif selected == "Author Analysis":

    st.title("👨‍🔬 Author Analysis")

    top_authors = (
        papers_df['authors']
        .value_counts()
        .head(15)
        .reset_index()
    )

    top_authors.columns = ['Author', 'Papers']

    fig = px.bar(
        top_authors,
        x='Papers',
        y='Author',
        orientation='h',
        title='Top Contributing Authors'
    )

    st.plotly_chart(fig, use_container_width=True)

    distribution_path = os.path.join(DATA_PATH, "author_distribution.png")

    if os.path.exists(distribution_path):
        st.markdown("---")
        st.subheader("Author Distribution")

        dist_img = Image.open(distribution_path)

        st.image(dist_img, use_container_width=True)

# =========================
# RESEARCH GAPS
# =========================

elif selected == "Research Gaps":

    st.title("⚠️ Research Gap Analysis")

    gap_html = os.path.join(DATA_PATH, "research_gap_growth.html")

    if os.path.exists(gap_html):
        with open(gap_html, "r", encoding="utf-8") as f:
            html_data = f.read()

        st.components.v1.html(html_data, height=700, scrolling=True)

    gap_csv = os.path.join(DATA_PATH, "research_gaps.csv")

    if os.path.exists(gap_csv):

        gaps_df = pd.read_csv(gap_csv)

        st.markdown("---")
        st.subheader("Top Potential Research Gaps")

        st.dataframe(
            gaps_df.sort_values(
                by="gap_score",
                ascending=False
            ).head(20),
            use_container_width=True
        )

        fig = px.scatter(
            gaps_df.head(100),
            x="doc_freq",
            y="gap_score",
            size="avg_tfidf",
            hover_name="keyword",
            title="Research Gap Opportunities"
        )

        st.plotly_chart(fig, use_container_width=True)


# =========================
# EMERGING DOMAIN DETECTION
# =========================

elif selected == "Emerging Domain Detection":

    st.title("🚀 Emerging Domain Detection")

    velocity_path = os.path.join(DATA_PATH, "keyword_velocity.png")

    if os.path.exists(velocity_path):

        velocity_img = Image.open(velocity_path)

        st.image(velocity_img, use_container_width=True)

    st.info(
        "This visualization highlights the fastest growing and declining research keywords over the last 5 years."
    )

# =========================
# PAPER EXPLORER
# =========================

elif selected == "Paper Explorer":

    st.title("📖 Research Paper Explorer")

    years = sorted(papers_df['year'].dropna().unique())

    selected_year = st.selectbox(
        "Select Year",
        years
    )

    filtered_df = papers_df[papers_df['year'] == selected_year]

    st.write(f"Total Papers: {len(filtered_df)}")

    search = st.text_input("Search by Title")

    if search:
        filtered_df = filtered_df[
            filtered_df['title'].str.contains(search, case=False, na=False)
        ]

    st.dataframe(
        filtered_df[
            ['title', 'authors', 'year', 'categories']
        ],
        use_container_width=True
    )
# =========================
# INTERACTIVE VISUALS
# =========================

elif selected == "Interactive Visuals":

    st.title("🌐 Interactive Visualizations")

    html_files = {
        "UMAP Paper Clusters": "umap_paper_clusters.html",
        "Topic Bubble Chart": "topic_bubble_chart.html",
        "Research Gap Growth": "research_gap_growth.html",
        "Topic Distribution": "lda_topic_distribution.html",
        "Topic Trends": "topic_trends_over_time.html"
    }

    selected_html = st.selectbox(
        "Choose Visualization",
        list(html_files.keys())
    )

    html_path = os.path.join(DATA_PATH, html_files[selected_html])

    if os.path.exists(html_path):
        with open(html_path, 'r', encoding='utf-8') as f:
            html_data = f.read()

        st.components.v1.html(html_data, height=800, scrolling=True)

Overwriting dashboard.py


In [24]:
!streamlit run dashboard.py &>/content/logs.txt &

In [25]:
!cat /content/logs.txt



2026-05-08 07:07:55.619 Uvicorn server started on 0.0.0.0:8504

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8504
  Network URL: http://172.28.0.12:8504
  External URL: http://136.118.205.193:8504



In [26]:
from pyngrok import ngrok

ngrok.set_auth_token("3DNjz9Huu7Nr8GdGxnJ8HObgx1d_KuYhwbTM46sgGJR6SBKa")

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://napkin-landed-award.ngrok-free.dev" -> "http://localhost:8501"
